<div dir="rtl">

# جلسه ۸: ساخت سیستم RAG

**Building a Retrieval-Augmented Generation System**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/08.%20Building_a_RAG_System.ipynb)

**دوره مهندسی LLM و هوش مصنوعی مولد** — کیوان محمودی

### اهداف این جلسه

- دیدن مشکل توهم مدل زبانی در پرسش‌های مربوط به اطلاعات اختصاصی یک سازمان
- ساخت یک سیستم RAG کامل: بازیابی، پرامپت مستند به منابع و تولید پاسخ
- ارجاع به منبع و امتناع از پاسخ وقتی جواب در اسناد نیست
- مدیریت بودجه توکن و ترتیب تکه‌ها در پرامپت
- RAG گفتگومحور با بازنویسی پرسش‌های وابسته به تاریخچه

> **سخت‌افزار:** این نوت‌بوک به GPU نیاز دارد. در Colab از منوی `Runtime → Change runtime type` گزینه **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl">

## ۱. چرا RAG؟

مدل زبانی هر چیزی را که در داده آموزشی‌اش بوده «می‌داند»، ولی:

- اطلاعات **اختصاصی** سازمان شما، مثل سیاست‌های فروشگاه، آیین‌نامه‌ها و قراردادها، را ندیده است.
- دانشش در **تاریخ مشخصی** متوقف شده است.
- وقتی جواب را نمی‌داند، اغلب پاسخی **باورپذیر ولی ساختگی** تولید می‌کند (**توهم**).

دو راه برای دادن دانش جدید به مدل وجود دارد: **فاین‌تیون** (جلسه ۱۰) و **RAG**. برای دانش واقعیت‌محور که مدام تغییر می‌کند، RAG تقریباً همیشه انتخاب اول است:

| | RAG | فاین‌تیون |
|---|---|---|
| به‌روزرسانی دانش | افزودن یا تغییر سند، فوری | آموزش دوباره |
| ارجاع به منبع | دارد | ندارد |
| هزینه شروع | کم | GPU و داده آموزشی |
| مناسب برای | دانش و واقعیت‌ها | سبک، قالب و رفتار |

</div>

<div dir="rtl">

## راه‌اندازی مدل زبانی

همه نوت‌بوک‌های این دوره از طریق API سازگار با OpenAI با مدل حرف می‌زنند. تابع `setup_llm` از فایل `llm_utils.py` دو حالت دارد:

- `setup_llm("ollama")`: مدل متن‌باز `qwen2.5:7b` را با Ollama روی همین ماشین اجرا می‌کند. اجرای اول چند دقیقه برای نصب و دانلود مدل طول می‌کشد. در Colab حتماً **T4 GPU** را فعال کنید.
- `setup_llm("api")`: به هر سرویس سازگار با OpenAI وصل می‌شود. نشانی، کلید و نام مدل را در بخش **Secrets** کولب (آیکون کلید در نوار کناری) با نام‌های `LLM_BASE_URL`، `LLM_API_KEY` و `LLM_MODEL` ذخیره کنید.

جزئیات کار این تابع در جلسه ۱ توضیح داده شده است.

</div>

In [ ]:
!pip -q install openai sentence-transformers
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

<div dir="rtl">

ابتدا ببینیم مدل بدون دسترسی به اسناد درباره نوین‌کالا چه می‌گوید:

</div>

In [ ]:
for question in ["هزینه ارسال سریع در نوین‌کالا چقدر است؟",
                 "گارانتی لپ‌تاپ‌های نوین‌کالا چند ماه است؟"]:
    print(question)
    print("  ->", chat(client, MODEL, question, temperature=0)[:300], "\n")

<div dir="rtl">

مدل یا عددی ساختگی می‌گوید یا پاسخ کلی می‌دهد. پاسخ درست، طبق اسناد، ۸۵ هزار تومان و ۱۸ ماه است. حالا RAG را می‌سازیم.

## ۲. بازیابی

تکه‌هایی را که در جلسه ۷ ساختیم دانلود می‌کنیم. این تکه‌ها نرمال شده‌اند و عنوان سند و بخش در ابتدای هر کدام آمده است. همان مدل امبدینگ و همان نرمال‌ساز را برای پرسش‌ها هم به کار می‌بریم.

</div>

In [ ]:
!pip -q install hazm

import json

import numpy as np
from hazm import Normalizer
from llm_utils import download_data
from sentence_transformers import SentenceTransformer

with open(download_data("novinkala_chunks.jsonl"), encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]

_normalizer = Normalizer(persian_numbers=False)
_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")


def normalize(text):
    return _normalizer.normalize(text).translate(_DIGITS)


class Retriever:
    """Dense retrieval over a list of chunks with multilingual E5."""

    def __init__(self, chunks, model_name="intfloat/multilingual-e5-base"):
        self.chunks = chunks
        self.model = SentenceTransformer(model_name)
        self.vectors = self.model.encode([f"passage: {c['text']}" for c in chunks],
                                         normalize_embeddings=True, batch_size=32)

    def search(self, query, k=4):
        q = self.model.encode([f"query: {normalize(query)}"], normalize_embeddings=True)[0]
        scores = self.vectors @ q
        top = np.argsort(-scores)[:k]
        return [{**self.chunks[i], "score": float(scores[i])} for i in top]


retriever = Retriever(chunks)
for hit in retriever.search("هزینه ارسال سریع چقدر است؟", k=3):
    print(f"{hit['score']:.3f}  {hit['id']}  {hit['text'][:90]}...")

<div dir="rtl">

## ۳. پرامپت مستند به منابع

پرامپت RAG سه بخش دارد: **دستورالعمل**، **منابع** و **پرسش**. نکته‌های کلیدی دستورالعمل:

1. فقط بر اساس منابع پاسخ بده، نه دانش قبلی.
2. اگر پاسخ در منابع نیست، صریحاً بگو. این مهم‌ترین قاعده برای جلوگیری از توهم است.
3. به منبع ارجاع بده، مثلاً `[2]`، تا کاربر و ما بتوانیم پاسخ را بررسی کنیم.
4. زبان و لحن پاسخ.

منابع را شماره‌گذاری می‌کنیم و هر کدام را در تگ جدا قرار می‌دهیم.

</div>

In [ ]:
RAG_SYSTEM = """You are the customer-support assistant of Novin Kala, an Iranian online store.
Answer the customer's question in Persian, using ONLY the numbered sources provided.
Rules:
- Every factual sentence must end with the number of its source in brackets, e.g. [2].
- If the sources do not contain the answer, reply exactly: «متأسفانه در اسناد نوین‌کالا پاسخی برای این پرسش پیدا نکردم.»
- Do not use outside knowledge and do not guess numbers, prices or dates.
- Mention conditions and exceptions that apply. Be concise and polite."""


def build_prompt(question, hits):
    sources = "\n\n".join(f"<source id=\"{i}\">\n{h['text']}\n</source>"
                           for i, h in enumerate(hits, 1))
    return f"{sources}\n\nCustomer question: {question}"


hits = retriever.search("هزینه ارسال سریع چقدر است؟", k=3)
print(build_prompt("هزینه ارسال سریع چقدر است؟", hits)[:700])

<div dir="rtl">

## ۴. اولین سیستم RAG

</div>

In [ ]:
import re


def rag_answer(question, k=4, verbose=True):
    hits = retriever.search(question, k=k)
    answer = chat(client, MODEL, build_prompt(question, hits), system=RAG_SYSTEM, temperature=0)
    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer) if 0 < int(n) <= len(hits)})
    if verbose:
        print(f"Q: {question}\nA: {answer}")
        for n in cited:
            print(f"   [{n}] {hits[n - 1]['id']} - {hits[n - 1]['title']} > {hits[n - 1]['section']}")
        print()
    return {"answer": answer, "hits": hits, "cited": cited}


for q in ["هزینه ارسال سریع چقدر است؟",
          "گارانتی لپ‌تاپ چند ماه است؟",
          "هدفون داخل گوشی رو باز کردم، می‌تونم پس بدم؟"]:
    rag_answer(q)

<div dir="rtl">

## ۵. آزمون‌های دشوار

یک سیستم RAG خوب فقط به پرسش‌های ساده جواب نمی‌دهد. باید در موارد زیر هم درست رفتار کند:

- **پرسش خارج از دامنه:** جواب در اسناد نیست.
- **پیش‌فرض غلط:** پرسش چیزی را فرض کرده که درست نیست. مثلاً ارسال سریع به تبریز وجود ندارد.
- **پرسش ترکیبی:** جواب در دو سند مختلف است.
- **زبان محاوره:** کلمات پرسش با کلمات سند فرق دارند.

</div>

In [ ]:
hard_questions = [
    "قیمت دلار امروز چنده؟",                                        # out of scope
    "هزینه ارسال سریع به تبریز چقدر است؟",                          # false premise
    "اگر یخچال را قسطی بخرم، پرداخت در محل هم دارد؟",              # two documents
    "رمزی که پیامک میشه تا کی اعتبار داره؟",                        # colloquial
]
for q in hard_questions:
    rag_answer(q)

<div dir="rtl">

پاسخ‌ها را با دقت بخوانید و برای هر کدام بپرسید: آیا درست است؟ آیا ارجاع‌ها واقعاً جمله را پشتیبانی می‌کنند؟ مدل‌های کوچک محلی گاهی قاعده امتناع را رعایت نمی‌کنند یا ارجاع غلط می‌دهند. در جلسه ۹ این خطاها را **اندازه** می‌گیریم.

## ۶. چند تکه و به چه ترتیب؟

- **k کوچک** (۱ یا ۲): اگر تکه درست در رتبه سوم باشد، از دست می‌رود.
- **k بزرگ** (۱۰ یا بیشتر): هزینه و تأخیر بیشتر، و تکه‌های بی‌ربط مدل را گیج می‌کنند.

پژوهش‌ها نشان داده‌اند مدل‌ها اطلاعات **وسط** یک ورودی طولانی را کمتر از ابتدا و انتهای آن استفاده می‌کنند. به این پدیده **Lost in the Middle** می‌گویند. پس مرتبط‌ترین تکه‌ها را در ابتدا قرار دهید و k را بی‌دلیل بزرگ نکنید.

بودجه توکن را هم باید کنترل کرد. تابع زیر تکه‌ها را تا رسیدن به سقف مشخصی از توکن اضافه می‌کند:

</div>

In [ ]:
def fit_to_budget(hits, max_tokens=1500):
    """Keep the best hits whose total length stays under the token budget."""
    tokenizer = retriever.model.tokenizer
    kept, total = [], 0
    for hit in hits:
        n = len(tokenizer(hit["text"]).input_ids)
        if total + n > max_tokens:
            break
        kept.append(hit)
        total += n
    return kept, total


kept, total = fit_to_budget(retriever.search("شرایط بازگشت کالا چیست؟", k=10), max_tokens=600)
print(f"{len(kept)} chunks, about {total} tokens")

<div dir="rtl">

## ۷. RAG گفتگومحور

در گفتگو، پرسش‌های بعدی معمولاً به قبلی‌ها وابسته‌اند:

> کاربر: ارسال سریع در کدام شهرها هست؟
> کاربر: **هزینه‌اش** چقدره؟

جستجوی «هزینه‌اش چقدره؟» به‌تنهایی معلوم نیست درباره چیست. راه‌حل رایج: پیش از بازیابی، از مدل می‌خواهیم با توجه به تاریخچه، پرسش را به یک **پرسش مستقل (Standalone Question)** بازنویسی کند.

</div>

In [ ]:
REWRITE_SYSTEM = (
    "Rewrite the customer's last message as a standalone question in Persian that can be "
    "understood without the conversation. Keep all details. Output only the rewritten question."
)


class ConversationalRAG:
    def __init__(self, k=4):
        self.history = []          # list of (question, answer)
        self.k = k

    def standalone(self, question):
        if not self.history:
            return question
        transcript = "\n".join(f"Customer: {q}\nAssistant: {a}" for q, a in self.history[-3:])
        return chat(client, MODEL, f"{transcript}\nCustomer: {question}",
                    system=REWRITE_SYSTEM, temperature=0).strip()

    def ask(self, question):
        query = self.standalone(question)
        result = rag_answer(query, k=self.k, verbose=False)
        self.history.append((question, result["answer"]))
        print(f"Q: {question}\n   (search: {query})\nA: {result['answer']}\n")
        return result


bot = ConversationalRAG()
bot.ask("ارسال سریع در کدام شهرها انجام می‌شود؟")
bot.ask("هزینه‌اش چقدره؟")
bot.ask("جمعه‌ها هم میارید؟")

<div dir="rtl">

بازنویسی پرسش یک فراخوانی اضافه به مدل است و تأخیر را بیشتر می‌کند. فقط زمانی لازم است که تاریخچه وجود دارد. مدیریت حافظه گفتگو را در جلسه ۱۵ کامل‌تر بررسی می‌کنیم.

## ۸. جمع‌بندی معماری

| جزء | انتخاب ما | جایگزین‌ها |
|---|---|---|
| تقسیم متن | بخش + عنوان، حداکثر ۶۰۰ کاراکتر | بازگشتی، معنایی |
| امبدینگ | multilingual-e5-base | bge-m3، امبدینگ API |
| شاخص | ضرب ماتریسی NumPy | FAISS، Chroma، Qdrant |
| بازیابی | چگال، top-4 | ترکیبی + Reranker (جلسه ۹) |
| تولید | qwen2.5:7b با ارجاع و امتناع | هر مدل سازگار با OpenAI |
| گفتگو | بازنویسی پرسش | حافظه خلاصه‌شده (جلسه ۱۵) |

## ۹. جمع‌بندی

- RAG دانش اختصاصی و به‌روز را بدون آموزش دوباره در اختیار مدل می‌گذارد و توهم را کاهش می‌دهد.
- پرامپت RAG باید منابع را شماره‌گذاری کند، ارجاع بخواهد و قاعده امتناع صریح داشته باشد.
- سیستم را با پرسش‌های دشوار بیازمایید: خارج از دامنه، پیش‌فرض غلط، ترکیبی و محاوره‌ای.
- k و بودجه توکن را کنترل کنید و مرتبط‌ترین تکه‌ها را اول بگذارید.
- در گفتگو، پیش از بازیابی، پرسش را مستقل کنید.

## ۱۰. تمرین

۱. قاعده امتناع را از `RAG_SYSTEM` حذف کنید و پرسش‌های بخش ۵ را دوباره اجرا کنید. چه تغییری می‌بینید؟

۲. تابع `rag_answer` را طوری تغییر دهید که اگر امتیاز بهترین تکه کمتر از یک آستانه بود، بدون فراخواندن مدل امتناع کند. آستانه مناسب را با چند پرسش داخل و خارج از دامنه پیدا کنید.

۳. یک «بررسی ارجاع» بسازید: برای هر جمله پاسخ که ارجاع دارد، از مدل بپرسید آیا منبع ارجاع‌داده‌شده آن جمله را پشتیبانی می‌کند یا نه.

۴. اسناد یک حوزه دیگر، مثلاً آیین‌نامه آموزشی یا راهنمای یک نرم‌افزار، را با خط لوله جلسه ۷ آماده کنید و با همین کد یک دستیار RAG برای آن بسازید.

</div>